# Create International Prize for Biology Awards

Creates OpenAlex award rows from the official Japan Society for the Promotion of Science (JSPS) International Prize for Biology pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/international_biology_prize_to_s3.py` to fetch the official JSPS pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:**
- Past-recipient table: `https://www.jsps.go.jp/english/e-biol/02_recipients.html`
- Prize amount/rules page: `https://www.jsps.go.jp/english/e-biol/01_outline.html`
- Newer official detail pages not yet on the summary table, including `awardee2024.html` and `awardee2025.html`

**S3 location:** `s3a://openalex-ingest/awards/international_biology_prize/international_biology_prize_recipients.parquet`

**Local full-source validation on 2026-05-26/27:** 41 official recipient rows, 1985-2025. Coverage: 100% title/year/recipient/research-field/affiliation/description/amount/currency. The summary table lists 1985-2023; the script also includes first-party 2024 and 2025 detail pages that already exist on the JSPS site.

**OpenAlex funder mapping:**
- Chosen funder_id: 4320334764
- Display name: Japan Society for the Promotion of Science
- Country: JP
- DOI/ROR: not present on the OpenAlex funder row
- Rationale: the International Prize for Biology is a JSPS award of recognition. This is operationally separate from the existing JSPS/KAKEN grants ingest at priority 10, with distinct provenance and native award IDs.

**Mapping summary:** One OpenAlex award row per official recipient. `amount=10000000` and `currency='JPY'` are populated for all rows from the official prize rule saying the prize consists of a medal and ten million yen. `funder_scheme` is the annual research field.

## Step 1: Load Raw Parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.international_biology_prize_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/international_biology_prize/international_biology_prize_recipients.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-26/27 found 41 rows.
SELECT COUNT(*) as total_international_biology_recipients
FROM openalex.awards.international_biology_prize_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.international_biology_prize_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, source_year, recipient_name, research_field, amount, currency, affiliation_raw
FROM openalex.awards.international_biology_prize_raw
ORDER BY source_year DESC
LIMIT 10;

## Step 1.5: Source, Money, and Key Scans

In [ ]:
%sql
SELECT column_name
FROM (DESCRIBE openalex.awards.international_biology_prize_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(recipient_name) AS rows_with_recipient,
    COUNT(research_field) AS rows_with_research_field,
    COUNT(affiliation_raw) AS rows_with_affiliation,
    COUNT(description) AS rows_with_description,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year,
    SUM(TRY_CAST(amount AS DOUBLE)) AS total_jpy
FROM openalex.awards.international_biology_prize_raw;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT CONCAT(source_year, ':', recipient_name)) AS distinct_year_recipient_keys
FROM openalex.awards.international_biology_prize_raw;

In [ ]:
%sql
SELECT source_year, recipient_name, research_field, landing_page_url
FROM openalex.awards.international_biology_prize_raw
ORDER BY TRY_CAST(source_year AS INT) DESC
LIMIT 8;

In [ ]:
%sql
SELECT currency, TRY_CAST(amount AS DOUBLE) AS amount, COUNT(*) AS rows
FROM openalex.awards.international_biology_prize_raw
GROUP BY currency, TRY_CAST(amount AS DOUBLE)
ORDER BY rows DESC;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for JSPS F4320334764'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320334764;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320334764;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.international_biology_prize_awards
USING delta
AS
WITH
funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320334764
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.international_biology_prize_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        NULLIF(TRIM(g.currency), '') as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'prize' as funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'International Prize for Biology') as funder_scheme,
        'international_biology_prize' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,
        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation_raw), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN funder_resolved f
)
SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 127

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'international_biology_prize' AND priority = 127;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    127 as priority
FROM openalex.awards.international_biology_prize_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_international_biology_awards
FROM openalex.awards.international_biology_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.international_biology_prize_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       lead_investigator.affiliation.name AS affiliation,
       landing_page_url
FROM openalex.awards.international_biology_prize_awards
ORDER BY start_year DESC
LIMIT 10;

In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids,
       COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.international_biology_prize_awards;

In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) AS cnt
FROM openalex.awards.international_biology_prize_awards
GROUP BY funder.display_name, funder_id;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(description) as has_description, COUNT(amount) as has_amount,
       COUNT(currency) as has_currency, COUNT(lead_investigator) as has_lead,
       COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.international_biology_prize_awards;

In [ ]:
%sql
SELECT currency, amount, COUNT(*) AS rows
FROM openalex.awards.international_biology_prize_awards
GROUP BY currency, amount
ORDER BY rows DESC;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) AS rows
FROM openalex.awards.international_biology_prize_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC;

In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'international_biology_prize' AND priority = 127
GROUP BY priority, provenance;